In [37]:
import pandas as pd
import numpy as np


Iterate through semester directories, read each tab-delimited log file, label it with its semester, and store the results in a dictionary.

In [38]:
import os
import pandas as pd

base_path = "Logs-206-201-Datashop"

datasets = {}

for semester in os.listdir(base_path):
    semester_path = os.path.join(base_path, semester)
    
    if os.path.isdir(semester_path):
        for file in os.listdir(semester_path):
            if file.endswith((".csv", ".tab", ".txt")):
                
                file_path = os.path.join(semester_path, file)
                
                df = pd.read_csv(file_path, sep="\t")
                df["Semester"] = semester
                
                datasets[semester] = df


In [39]:
datasets.keys()


dict_keys(['F21', 'F23', 'F22', 'W24'])

In [40]:
datasets["W24"].head()
datasets["W24"].shape
datasets["W24"].columns

datasets.values()


dict_values([                               Time  Anon Student Id  \
0               2021-08-24 16:02:07               22   
1               2021-08-24 16:02:21               22   
2               2021-08-24 16:02:33               22   
3               2021-08-24 16:02:35               22   
4               2021-08-24 16:02:41               22   
...                             ...              ...   
1285964  2022-01-12 04:45:20.404335               79   
1285965  2022-01-12 04:45:31.000000               79   
1285966  2022-01-13 15:13:33.847926               32   
1285967  2022-01-13 15:13:43.881009               32   
1285968  2022-01-13 18:17:41.000000              108   

                                Action                          Problem Name  \
0                               donate                             no params   
1                                 view                   py4e-int/index.html   
2                                 view           py4e-int/intro/toctree.ht

Creates one consolidated dataset containing all student interaction logs across semesters.

In [22]:
combined_df = pd.concat(datasets.values(), ignore_index=True)
combined_df.shape


(5338385, 17)

Clean column names.


In [30]:
combined_df.columns = (
    combined_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
)



Collapses feedback categories into three standardized outcome types.

In [43]:
combined_df["feedback_classification"].value_counts(normalize=True)


feedback_classification
success                                                                                                                                                                                                                                    3.174070e-01
Correct                                                                                                                                                                                                                                    2.766578e-01
Incorrect                                                                                                                                                                                                                                  2.619127e-01
SyntaxError                                                                                                                                                                                                                             

In [34]:
def simplify_outcome(x):
    if x in ["Correct", "success"]:
        return "Correct"
    elif x == "Incorrect":
        return "Incorrect"
    else:
        return "Error"

combined_df["outcome"] = combined_df["feedback_classification"].apply(simplify_outcome)


In [35]:
combined_df["outcome"].value_counts(normalize=True)


outcome
Error        0.723628
Correct      0.191807
Incorrect    0.084565
Name: proportion, dtype: float64

Drops interaction logs that cannot be tied to a specific student.

In [32]:
combined_df = combined_df.dropna(subset=["anon_student_id"])


In [44]:
combined_df.shape

(5338385, 18)

In [42]:
combined_df.columns

Index(['time', 'anon_student_id', 'action', 'problem_name', 'class',
       'level_chapter', 'level_subchapter', 'session_id', 'problem_view',
       'selection', 'input', 'feedback_text', 'feedback_classification',
       'cf_code', 'cf_week_no', 'cf_institution', 'semester', 'outcome'],
      dtype='str')

In [48]:
combined_df.cf_institution

0          university
1          university
2          university
3          university
4          university
              ...    
5338380    university
5338381    university
5338382    university
5338383    university
5338384    university
Name: cf_institution, Length: 5338385, dtype: str

In [ ]:
combined_df = combined_df.drop(columns=["cf_institution"])


In [51]:
combined_df.columns

Index(['time', 'anon_student_id', 'action', 'problem_name', 'class',
       'level_chapter', 'level_subchapter', 'session_id', 'problem_view',
       'selection', 'input', 'feedback_text', 'feedback_classification',
       'cf_code', 'cf_week_no', 'semester', 'outcome'],
      dtype='str')

In [68]:
combined_df["cf_code"].isna().mean()


np.float64(0.850755425095792)

In [70]:
combined_df.action.value_counts()

action
interaction                                  942486
run                                          337782
view                                         325243
viewassignment                               235153
doAssignment                                 154387
                                              ...  
move|0_0-8_0-4_0-6_0-7_0-5_0|1_0-2_3_0|c0         1
move|0_0-4_0-6_0-7_0-5_0|1_0-2_3_0-8_0|c0         1
move|0_0-6_0-7_0-5_0|1_0-2_3_0-4_0-8_0|c0         1
slide:8/21/2024, 11:02:54 PM                      1
slide:8/21/2024, 11:03:37 PM                      1
Name: count, Length: 1045017, dtype: int64

In [71]:
def simplify_action(x):
    if x in ["run", "view", "interaction", "viewassignment", "doAssignment"]:
        return x
    else:
        return "other"

combined_df["action_simple"] = combined_df["action"].apply(simplify_action)


In [73]:
combined_df.action_simple.value_counts()

action_simple
other             3343334
interaction        942486
run                337782
view               325243
viewassignment     235153
doAssignment       154387
Name: count, dtype: int64

In [82]:
combined_df.groupby("problem_name")["outcome"].value_counts(normalize=True)


problem_name                                                          outcome  
/ns/books/published/Fall22-SI206/classes-basics/MultipleClasses.html  Error        1.000000
/ns/books/published/Fall22-SI206/conditional/pogil.html               Error        1.000000
/ns/books/published/Fall22-SI206/files/csv-file-group.html            Error        1.000000
/ns/books/published/Fall22-SI206/files/csv-reader-file-group.html     Error        1.000000
/ns/books/published/Fall22-SI206/files/file-group-basics.html         Error        1.000000
                                                                                     ...   
xml_parse_write_code_note_data_ac                                     Correct      0.119336
xml_person_attributes_clicka                                          Incorrect    0.508088
                                                                      Correct      0.491912
xml_person_self_closing_tags_clicka                                   Incorrect    0.548913


Restrict to rows that represent real submission events.

In [83]:
analysis_df = combined_df[combined_df["action"] == "run"]


In [84]:
analysis_df.groupby("problem_name")["outcome"].value_counts(normalize=True)


problem_name                       outcome
01-active-hello                    Error      1.000000
02-ac-1-vars2                      Error      0.934211
                                   Correct    0.065789
02-ac-10-input                     Error      1.000000
02-ac-10-names1                    Error      1.000000
                                                ...   
xml_parse_person_wihtout_attr_ac1  Correct    0.004405
xml_parse_write_code_book_data_ac  Error      0.674389
                                   Correct    0.325611
xml_parse_write_code_note_data_ac  Error      0.772500
                                   Correct    0.227500
Name: proportion, Length: 2053, dtype: float64

In [86]:
df.columns

Index(['Time', 'Anon Student Id', 'Action', 'Problem Name', 'Class',
       'Level (Chapter)', 'Level (SubChapter)', 'Session Id', 'Problem View',
       'Selection', 'Input', 'Feedback Text', 'Feedback Classification',
       'CF (Code)', 'CF (Week No)', 'CF (Institution)', 'Semester'],
      dtype='str')